In [1]:
import sys, os
from pathlib import Path
import polars as pl

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("Directorio de trabajo:", Path.cwd())

Directorio de trabajo: e:\Pruebas Tecnicas\Adidas\Manager Omnichannel\BusinessCase_ManagerOmniAnalytics 1\Candidate_Package


In [2]:
fact_orders = pl.read_parquet("data/silver/fact_orders.parquet")
print(fact_orders.shape)
fact_orders.columns

(1425316, 16)


['channel',
 'consumer_id',
 'order_id',
 'country',
 'year_month',
 'category',
 'product_id',
 'qty_ordered',
 'qty_sold',
 'unit_price',
 'currency',
 'revenue_local',
 'order_status',
 'adiclub',
 'revenue_eur',
 'unit_price_eur']

In [3]:
from src import kpis

gold_country_month = kpis.build_kpi_table(fact_orders, group_by=["country", "year_month"])
gold_channel_category = kpis.build_kpi_table(fact_orders, group_by=["channel", "category"])
gold_country_channel = kpis.build_kpi_table(fact_orders, group_by=["country", "channel"])

print("País x Mes:", gold_country_month.shape)
print("Canal x Categoría:", gold_channel_category.shape)
print("País x Canal:", gold_country_channel.shape)

País x Mes: (72, 5)
Canal x Categoría: (20, 5)
País x Canal: (30, 5)


In [4]:
print(gold_country_month.head(10))
print()
print(gold_channel_category.head(10))
print()
print(gold_country_channel.head(10))

shape: (10, 5)
┌─────────┬────────────┬───────────────────┬─────────────┬──────────────────┐
│ country ┆ year_month ┆ revenue_bruto_eur ┆ return_rate ┆ fulfillment_rate │
│ ---     ┆ ---        ┆ ---               ┆ ---         ┆ ---              │
│ str     ┆ str        ┆ f64               ┆ f64         ┆ f64              │
╞═════════╪════════════╪═══════════════════╪═════════════╪══════════════════╡
│ AR      ┆ 2025-01    ┆ 4.5381e6          ┆ 0.125762    ┆ 0.781004         │
│ AR      ┆ 2025-02    ┆ 4.1660e6          ┆ 0.125557    ┆ 0.783717         │
│ AR      ┆ 2025-03    ┆ 5.2314e6          ┆ 0.128945    ┆ 0.7842           │
│ AR      ┆ 2025-04    ┆ 5.5491e6          ┆ 0.126278    ┆ 0.779758         │
│ AR      ┆ 2025-05    ┆ 6.1993e6          ┆ 0.128073    ┆ 0.775057         │
│ AR      ┆ 2025-06    ┆ 6.5136e6          ┆ 0.124029    ┆ 0.781824         │
│ AR      ┆ 2025-07    ┆ 5.4502e6          ┆ 0.12685     ┆ 0.778494         │
│ AR      ┆ 2025-08    ┆ 5.8542e6          ┆ 0.12

In [5]:
GOLD_DIR = Path("data/gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

gold_country_month.write_parquet(GOLD_DIR / "kpis_country_month.parquet")
gold_channel_category.write_parquet(GOLD_DIR / "kpis_channel_category.parquet")
gold_country_channel.write_parquet(GOLD_DIR / "kpis_country_channel.parquet")

print("Archivos en gold:")
for f in GOLD_DIR.glob("*.parquet"):
    print(" -", f.name)

Archivos en gold:
 - kpis_channel_category.parquet
 - kpis_country_channel.parquet
 - kpis_country_month.parquet


In [6]:
print(fact_orders.group_by("channel").agg(pl.col("adiclub").null_count().alias("adiclub_nulls"), pl.len()))

shape: (5, 3)
┌───────────┬───────────────┬────────┐
│ channel   ┆ adiclub_nulls ┆ len    │
│ ---       ┆ ---           ┆ ---    │
│ str       ┆ u32           ┆ u32    │
╞═══════════╪═══════════════╪════════╡
│ retail    ┆ 14177         ┆ 707000 │
│ app       ┆ 0             ┆ 202000 │
│ wholesale ┆ 66866         ┆ 66866  │
│ ecomm     ┆ 8218          ┆ 404000 │
│ franchise ┆ 935           ┆ 45450  │
└───────────┴───────────────┴────────┘


In [7]:
gold_adiclub = kpis.build_kpi_table(
    fact_orders.filter(pl.col("adiclub").is_not_null()),
    group_by=["adiclub"]
)
print(gold_adiclub)

shape: (2, 4)
┌─────────┬───────────────────┬─────────────┬──────────────────┐
│ adiclub ┆ revenue_bruto_eur ┆ return_rate ┆ fulfillment_rate │
│ ---     ┆ ---               ┆ ---         ┆ ---              │
│ bool    ┆ f64               ┆ f64         ┆ f64              │
╞═════════╪═══════════════════╪═════════════╪══════════════════╡
│ false   ┆ 1.1682e8          ┆ 0.127746    ┆ 0.780167         │
│ true    ┆ 1.9260e8          ┆ 0.127704    ┆ 0.78014          │
└─────────┴───────────────────┴─────────────┴──────────────────┘


In [8]:
gold_adiclub_channel = kpis.build_kpi_table(
    fact_orders.filter(pl.col("adiclub").is_not_null()),
    group_by=["channel", "adiclub"]
)
print(gold_adiclub_channel.sort(["channel", "adiclub"]))

shape: (7, 5)
┌───────────┬─────────┬───────────────────┬─────────────┬──────────────────┐
│ channel   ┆ adiclub ┆ revenue_bruto_eur ┆ return_rate ┆ fulfillment_rate │
│ ---       ┆ ---     ┆ ---               ┆ ---         ┆ ---              │
│ str       ┆ bool    ┆ f64               ┆ f64         ┆ f64              │
╞═══════════╪═════════╪═══════════════════╪═════════════╪══════════════════╡
│ app       ┆ true    ┆ 4.9427e7          ┆ 0.126572    ┆ 0.779106         │
│ ecomm     ┆ false   ┆ 3.3678e7          ┆ 0.129736    ┆ 0.780367         │
│ ecomm     ┆ true    ┆ 4.1293e7          ┆ 0.12738     ┆ 0.780123         │
│ franchise ┆ false   ┆ 5.2339e6          ┆ 0.119736    ┆ 0.779415         │
│ franchise ┆ true    ┆ 6.4951e6          ┆ 0.129859    ┆ 0.781944         │
│ retail    ┆ false   ┆ 7.7910e7          ┆ 0.127423    ┆ 0.7801           │
│ retail    ┆ true    ┆ 9.5387e7          ┆ 0.128284    ┆ 0.780582         │
└───────────┴─────────┴───────────────────┴─────────────┴─────

In [9]:
comparison = (
    fact_orders
    .filter(pl.col("adiclub").is_not_null() & pl.col("order_status").is_in(["delivered", "returned"]))
    .group_by("adiclub")
    .agg([
        pl.col("revenue_eur").fill_null(0).sum().alias("revenue_total"),
        pl.len().alias("num_lineas"),
        pl.col("order_id").n_unique().alias("num_ordenes_unicas"),
    ])
    .with_columns([
        (pl.col("revenue_total") / pl.col("num_ordenes_unicas")).alias("avg_revenue_por_orden"),
    ])
)
print(comparison)

shape: (2, 5)
┌─────────┬───────────────┬────────────┬────────────────────┬───────────────────────┐
│ adiclub ┆ revenue_total ┆ num_lineas ┆ num_ordenes_unicas ┆ avg_revenue_por_orden │
│ ---     ┆ ---           ┆ ---        ┆ ---                ┆ ---                   │
│ bool    ┆ f64           ┆ u32        ┆ u32                ┆ f64                   │
╞═════════╪═══════════════╪════════════╪════════════════════╪═══════════════════════╡
│ false   ┆ 1.1682e8      ┆ 397668     ┆ 325001             ┆ 359.448962            │
│ true    ┆ 1.9260e8      ┆ 644076     ┆ 553504             ┆ 347.969546            │
└─────────┴───────────────┴────────────┴────────────────────┴───────────────────────┘


In [10]:
by_channel = (
    fact_orders
    .filter(pl.col("adiclub").is_not_null() & pl.col("order_status").is_in(["delivered", "returned"]))
    .group_by(["channel", "adiclub"])
    .agg([
        pl.col("order_id").n_unique().alias("num_ordenes"),
    ])
    .sort(["channel", "adiclub"])
)
print(by_channel)

shape: (7, 3)
┌───────────┬─────────┬─────────────┐
│ channel   ┆ adiclub ┆ num_ordenes │
│ ---       ┆ ---     ┆ ---         │
│ str       ┆ bool    ┆ u32         │
╞═══════════╪═════════╪═════════════╡
│ app       ┆ true    ┆ 155938      │
│ ecomm     ┆ false   ┆ 68800       │
│ ecomm     ┆ true    ┆ 84146       │
│ franchise ┆ false   ┆ 15387       │
│ franchise ┆ true    ┆ 19023       │
│ retail    ┆ false   ┆ 240814      │
│ retail    ┆ true    ┆ 294397      │
└───────────┴─────────┴─────────────┘


In [11]:
no_member_orders = 325001  # ya lo tenemos de comparison
member_orders = 553504
member_avg_ticket = 347.969546

# Si los no-miembros compraran con la misma frecuencia relativa que los miembros
# (asumiendo que convertir a X% de no-miembros movería su frecuencia al nivel de miembros)
uplift_ratio = member_orders / no_member_orders  # ya sabemos ~1.7, pero calculémoslo por canal para ser más precisos

print(f"Ratio de frecuencia miembro/no-miembro: {uplift_ratio:.2f}x")

# Ejemplo de impacto: si el 10% de la base no-miembro se convirtiera a un
# comportamiento de frecuencia tipo miembro
potential_new_orders = no_member_orders * 0.10 * (uplift_ratio - 1)
potential_revenue = potential_new_orders * member_avg_ticket
print(f"Revenue potencial estimado (10% de no-miembros migran a frecuencia de miembro): €{potential_revenue:,.0f}")

Ratio de frecuencia miembro/no-miembro: 1.70x
Revenue potencial estimado (10% de no-miembros migran a frecuencia de miembro): €7,951,209


In [12]:
GOLD_DIR = Path("data/gold")

comparison.write_parquet(GOLD_DIR / "kpis_adiclub_comparison.parquet")

by_channel.write_parquet(GOLD_DIR / "kpis_adiclub_by_channel.parquet")

print("Archivos en gold:")
for f in GOLD_DIR.glob("*.parquet"):
    print(" -", f.name)

Archivos en gold:
 - kpis_adiclub_by_channel.parquet
 - kpis_adiclub_comparison.parquet
 - kpis_channel_category.parquet
 - kpis_country_channel.parquet
 - kpis_country_month.parquet
